In [4]:
from pathlib import Path

from scipy.stats import wilcoxon
import pandas as pd
from pandas import IndexSlice

from config import PATHS

In [9]:
# on server:
# in_path = PATHS.per_model_comparison_table.pickle

# local:
in_path = Path('~/thesis_files/statistical_results/.per_model.pkl').expanduser()
out_path = Path('~/Developer/MastersThesis/src/tables/model_comparison.tex').expanduser()

In [10]:
metrics = pd.read_pickle(in_path)
metrics

metric                 best_threshold            rel_tifw            \
split                           train      test     train      test   
patient       model                                                   
competition-1 CNN            0.730252  0.348021  0.150953  0.407381   
              ensemble       0.548969  0.420712  0.368110  0.494720   
competition-2 CNN            0.592629  0.503402  0.196632  0.330344   
              ensemble       0.604010  0.561291  0.323201  0.397423   
competition-3 CNN            0.779933  0.696605  0.142371  0.263264   
              ensemble       0.654698  0.622529  0.217113  0.285408   
U002-DE01-01  CNN            0.582112  0.634063  0.226472  0.165048   
              ensemble       0.567867  0.544376  0.266903  0.239180   
U002-DE01-03  CNN            0.780604  0.674600  0.107185  0.373027   
              ensemble       0.574479  0.525727  0.186428  0.426924   
U002-DE01-04  CNN            0.599625  0.559766  0.269078  0.472266   
              ensemble       0.526974  0.505274  0.430727  0.600760   
U002-DE01-05  CNN            0.756170  0.598670  0.065411  0.217563   
              ensemble       0.587826  0.586498  0.171846  0.189639   
U002-DE01-07  CNN            0.736176  0.701969  0.034486  0.183846   
              ensemble       0.545029  0.536581  0.282319  0.457661   
U002-DE01-12  CNN            0.666480  0.465285  0.072165  0.394819   
              ensemble       0.563852  0.561054  0.420331  0.482148   
U002-DE01-15  CNN            0.967000  0.868920  0.047427  0.345278   
              ensemble       0.561869  0.539433  0.315991  0.413938   
U002-DE01-16  CNN            0.812213  0.752847  0.130959  0.350571   
              ensemble       0.563557  0.559030  0.168770  0.215980   
U002-DE01-17  CNN            0.667585  0.528467  0.156727  0.358979   
              ensemble       0.609517  0.604358  0.345519  0.305847   

metric                 rel_szrs_pred           event_based_f1            \
split                          train      test          train      test   
patient       model                                                       
competition-1 CNN           0.794118  0.541667       0.820664  0.565998   
              ensemble      0.852941  0.500000       0.725961  0.502626   
competition-2 CNN           0.967742  0.863636       0.877928  0.754376   
              ensemble      0.838710  0.636364       0.749106  0.619009   
competition-3 CNN           0.945946  0.846154       0.899625  0.787663   
              ensemble      0.837838  0.961538       0.809431  0.819874   
U002-DE01-01  CNN           0.880342  0.794872       0.823486  0.814419   
              ensemble      0.803419  0.782051       0.766649  0.771290   
U002-DE01-03  CNN           0.852941  0.666667       0.872423  0.646211   
              ensemble      0.735294  0.444444       0.772455  0.500630   
U002-DE01-04  CNN           0.783784  0.560000       0.756430  0.543388   
              ensemble      0.729730  0.400000       0.639591  0.399620   
U002-DE01-05  CNN           0.800000  0.857143       0.862073  0.818088   
              ensemble      0.775000  0.785714       0.800696  0.797847   
U002-DE01-07  CNN           1.000000  0.666667       0.982454  0.733875   
              ensemble      0.857143  0.833333       0.781237  0.657059   
U002-DE01-12  CNN           0.777778  0.666667       0.846206  0.634438   
              ensemble      0.833333  0.750000       0.683732  0.612672   
U002-DE01-15  CNN           0.769231  0.700000       0.851141  0.676604   
              ensemble      0.846154  0.600000       0.756490  0.592949   
U002-DE01-16  CNN           1.000000  0.625000       0.929932  0.636981   
              ensemble      0.750000  0.750000       0.788529  0.766633   
U002-DE01-17  CNN           0.833333  0.777778       0.838274  0.702808   
              ensemble      0.833333  0.777778       0.733157  0.733590   

metric                 precision              recall    

In [11]:
ALPHA = 0.05
metric_names = ['rel_tifw', 'rel_szrs_pred', 'event_based_f1', 'roc_auc']
splits = ['train', 'test']

results = {}
for metric_name in metric_names:
    row = {}
    for split in splits:
        metric = metrics[(metric_name, split)]

        cnn = metric.loc[IndexSlice[:, 'CNN']]
        ensemble = metric.loc[IndexSlice[:, 'ensemble']]

        stat, p = wilcoxon(cnn, ensemble)
        row[(split, 'stat')] = stat
        row[(split, 'p')] = p
        row[(split, 'significant')] = p < ALPHA
    results[metric_name] = row

table = pd.DataFrame.from_dict(results, orient='index')
table.columns = pd.MultiIndex.from_tuples(table.columns, names=['split', 'metric'])

table

split          train                        test                      
metric          stat         p significant  stat         p significant
rel_tifw         0.0  0.000488        True  16.0  0.077148       False
rel_szrs_pred   12.5  0.070312       False  26.0  0.577148       False
event_based_f1   0.0  0.000488        True  16.0  0.077148       False
roc_auc          0.0  0.000488        True  21.0  0.176270       False

In [4]:
#### Convert to LaTeX

In [12]:
t = table.copy()
for col in [('train', 'significant'), ('test', 'significant')]:
    t[col] = t[col].map({True: r'$\checkmark$', False: r'$\times$'}).astype(str)

t = t.rename(
    index={
        'rel_tifw': 'Relative TIFW',
        'rel_szrs_pred': 'EB sensitivity',
        'event_based_f1': 'EB score',
        'roc_auc': 'ROC AUC'
    },
    columns={
        'train': 'Train',
        'test': 'Test',
        'p': 'p-value',
        'stat': 'z-statistic'
    },
)
t = t.rename_axis(columns={'split': 'Partition', 'metric': 'Metric'})

t

Partition            Train                                Test            \
Metric         z-statistic   p-value   significant z-statistic   p-value   
relative TIFW          0.0  0.000488  $\checkmark$        16.0  0.077148   
EB sensitivity        12.5  0.070312      $\times$        26.0  0.577148   
EB score               0.0  0.000488  $\checkmark$        16.0  0.077148   
ROC AUC                0.0  0.000488  $\checkmark$        21.0  0.176270   

Partition                   
Metric         significant  
relative TIFW     $\times$  
EB sensitivity    $\times$  
EB score          $\times$  
ROC AUC           $\times$

In [30]:
styled = t.style.format({
    ('Train', 'z-statistic'): '{:.1f}',
    ('Test', 'z-statistic'): '{:.1f}',
    ('Train', 'p-value'): '{:.5f}',
    ('Test', 'p-value'): '{:.5f}',
})
styled

In [32]:
latex = styled.to_latex(
    column_format='lrrrrrr',
    multicol_align='c',
    hrules=True
)
# styled.to_latex(out_path)
print(latex)

\begin{tabular}{lrrrrrr}
\toprule
Partition & \multicolumn{3}{c}{Train} & \multicolumn{3}{c}{Test} \\
Metric & z-statistic & p-value & significant & z-statistic & p-value & significant \\
\midrule
relative TIFW & 0.0 & 0.00049 & $\checkmark$ & 16.0 & 0.07715 & $\times$ \\
EB sensitivity & 12.5 & 0.07031 & $\times$ & 26.0 & 0.57715 & $\times$ \\
EB score & 0.0 & 0.00049 & $\checkmark$ & 16.0 & 0.07715 & $\times$ \\
ROC AUC & 0.0 & 0.00049 & $\checkmark$ & 21.0 & 0.17627 & $\times$ \\
\bottomrule
\end{tabular}

